# Calibrated Stacked Ensembles for Explainable Credit Risk Scoring
Reproduces every number and figure in the 3ICT 2026 paper. Run all cells in order (Runtime > Run all).

**Everything lives under one Google Drive folder, so nothing is lost when the session closes:**

`/content/drive/MyDrive/credit_risk_3ict/`
- `code/` : `pipeline_revision.py`, `analyze.py`, `shap_figs.py`, `loss_sim.py`, `verify_paper.py`, `paper.tex`, `south_german_credit.asc`
- `data/` : prepared CSV files (created by this notebook)
- `out/`, `out_shap/`, `figures/`, `logs/` : results (created by this notebook)

Finished datasets are skipped on re-run. If Colab disconnects mid-run, run all cells again. Keep this tab open during the Taiwan run (it is the longest).

In [ ]:
# Cell 1: mount Drive, define the single project root, create folders, check inputs
import os, sys, shutil, subprocess, time
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("not running in Colab, skipping Drive mount")

BASE = '/content/drive/MyDrive/credit_risk_3ict'
CODE, DATA, OUT, OUT_SHAP, FIG, LOGS = (f'{BASE}/{d}' for d in ['code', 'data', 'out', 'out_shap', 'figures', 'logs'])
for d in [CODE, DATA, OUT, OUT_SHAP, FIG, LOGS]:
    os.makedirs(d, exist_ok=True)
os.chdir(BASE)
os.environ['CR_BASE'] = BASE      # the scripts read this to find data/ and write out/, out_shap/

need = ['pipeline_revision.py', 'analyze.py', 'shap_figs.py', 'loss_sim.py', 'verify_paper.py', 'south_german_credit.asc']
missing = [f for f in need if not os.path.exists(f'{CODE}/{f}')]
print('project root:', BASE)
print('files in code/:', sorted(os.listdir(CODE)))
assert not missing, f'Upload these to {CODE} and run again: {missing}'

def run(cmd, log):
    """Run a shell command from BASE, stream output, save it to logs/, stop on failure."""
    print('$', cmd, flush=True)
    with open(f'{LOGS}/{log}', 'w') as lf:
        p = subprocess.Popen(cmd, shell=True, cwd=BASE, env=os.environ.copy(),
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in p.stdout:
            sys.stdout.write(line); lf.write(line)
        p.wait()
    assert p.returncode == 0, f'command failed (exit {p.returncode}), see logs/{log}'

In [ ]:
# Cell 2: install packages and record exact versions in logs/package_versions.txt
run('pip -q install xgboost lightgbm catboost shap tabulate ucimlrepo', 'pip_install.log')
v = subprocess.run("pip freeze | grep -i -E '^(numpy|pandas|scipy|scikit-learn|xgboost|lightgbm|catboost|tensorflow|shap|matplotlib|ucimlrepo)=='",
                   shell=True, capture_output=True, text=True).stdout
open(f'{LOGS}/package_versions.txt', 'w').write(f'python {sys.version}\n{v}')
print(v)

In [ ]:
# Cell 3: prepare the South German Credit data  ->  data/GermanCredit.csv
import pandas as pd
d = pd.read_csv(f'{CODE}/south_german_credit.asc', sep=r'\s+')
d = d.rename(columns={'kredit': 'credit_risk'})          # 1 = good, 0 = bad
numeric = ['laufzeit', 'hoehe', 'alter', 'rate', 'wohnzeit', 'bishkred', 'pers']
for c in d.columns:
    if c not in numeric + ['credit_risk']:
        d[c] = d[c].astype(str)                          # categorical, one-hot encoded by the pipeline
d.to_csv(f'{DATA}/GermanCredit.csv', index=False)
print(d.shape, d['credit_risk'].value_counts().to_dict())
assert d.shape == (1000, 21) and d['credit_risk'].value_counts().to_dict() == {1: 700, 0: 300}, 'unexpected German data'

In [ ]:
# Cell 4: download the Taiwan credit-card default data (UCI id 350)  ->  data/taiwan_credit.csv
tw = f'{DATA}/taiwan_credit.csv'
if not os.path.exists(tw):
    from ucimlrepo import fetch_ucirepo
    t = fetch_ucirepo(id=350)
    X = t.data.features.copy()
    X['target'] = t.data.targets.iloc[:, 0].values
    X.to_csv(tw, index=False)
X = pd.read_csv(tw)
print(X.shape, 'default rate', round(X['target'].mean(), 4), list(X.columns)[:5])
assert X.shape == (30000, 24) and abs(X['target'].mean() - 0.2212) < 0.001, 'unexpected Taiwan data'

In [ ]:
# Cell 5: run the simulated tariff-stress portfolio (skipped if already finished)
if os.path.exists(f'{OUT}/synth_fold_metrics.csv'):
    print('synth already done, skipping')
else:
    run(f'python {CODE}/pipeline_revision.py --data synth --repeats 3 --out {OUT}', 'synth.log')

In [ ]:
# Cell 6: run the South German Credit data
if os.path.exists(f'{OUT}/german_fold_metrics.csv'):
    print('german already done, skipping')
else:
    run(f'python {CODE}/pipeline_revision.py --data german --repeats 3 --german_path {DATA}/GermanCredit.csv --out {OUT}', 'german.log')

In [ ]:
# Cell 7: run the Taiwan data (longest; keep the tab open)
if os.path.exists(f'{OUT}/taiwan_fold_metrics.csv'):
    print('taiwan already done, skipping')
else:
    run(f'python {CODE}/pipeline_revision.py --data taiwan --repeats 2 --n_max 10000 --taiwan_path {DATA}/taiwan_credit.csv --out {OUT}', 'taiwan.log')

In [ ]:
# Cell 8: tables, paired tests, reliability diagrams  ->  out/results.md, out/reliability_*.png
run(f'python {CODE}/analyze.py', 'analyze.log')

In [ ]:
# Cell 9: SHAP figures  ->  out_shap/ (manuscript Figures 3 and 4)
if os.path.exists(f'{OUT_SHAP}/shap_rankings.json'):
    print('SHAP already done, skipping')
else:
    run(f'python {CODE}/shap_figs.py', 'shap.log')

In [ ]:
# Cell 10: portfolio loss simulation (paper Section VI-F)  ->  out/loss_sim.json
run(f'python {CODE}/loss_sim.py', 'loss_sim.log')

In [ ]:
# Cell 11: collect the deliverables into figures/
import glob
for f in glob.glob(f'{OUT}/reliability_*.png') + glob.glob(f'{OUT_SHAP}/*.png'):
    shutil.copy(f, FIG)
shutil.copy(f'{OUT}/results.md', f'{FIG}/results.md')
print(sorted(os.listdir(FIG)))
print(open(f'{OUT}/results.md').read()[:1800])

In [ ]:
# Cell 12: check the paper against these results (needs code/paper.tex)
if os.path.exists(f'{CODE}/paper.tex'):
    run(f'python {CODE}/verify_paper.py {CODE}/paper.tex {OUT}', 'verify.log')
else:
    print(f'Upload paper.tex to {CODE} to run the paper check')

In [ ]:
# Cell 13: final summary of what is saved on Drive
for root, dirs, files in os.walk(BASE):
    depth = root[len(BASE):].count(os.sep)
    print('  ' * depth + os.path.basename(root) + '/')
    for f in sorted(files):
        print('  ' * (depth + 1) + f)